### Cốt lõi của thuật toán (DFS)

In [2]:
class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

class Problem:
    def __init__(self, start_state, target_state, get_moves, apply_move):
        self.start_state = start_state
        self.target_state = target_state
        self.get_moves = get_moves
        self.apply_move = apply_move

def is_goal(puzzle, cur_state):
    if callable(puzzle.target_state):
        return puzzle.target_state(cur_state)
    return cur_state == puzzle.target_state

def Stack():
    return []

def push(wait_stack, item):
    wait_stack.append(item)

def is_empty(wait_stack):
    return len(wait_stack) == 0

def pop(wait_stack):
    if is_empty(wait_stack):
        return None
    last_item = wait_stack[-1]
    del wait_stack[-1]
    return last_item

def child_node(puzzle, prev_node, move):
    next_state = puzzle.apply_move(prev_node.state, move)
    return Node(state=next_state, parent=prev_node, action=move)

def solution(goal_node):
    trace = []
    curr = goal_node
    while curr is not None:
        trace.append(curr.state)
        curr = curr.parent
    trace.reverse()
    return trace

def DFS(puzzle):
    root = Node(puzzle.start_state)

    if is_goal(puzzle, root.state):
        return solution(root)

    wait_stack = Stack()
    push(wait_stack, root)
    in_stack = {root.state}
    seen_states = set()

    while not is_empty(wait_stack):
        cur_node = pop(wait_stack)
        in_stack.discard(cur_node.state)
        seen_states.add(cur_node.state)

        for move in puzzle.get_moves(cur_node.state):
            next_node = child_node(puzzle, cur_node, move)

            if next_node.state not in seen_states and next_node.state not in in_stack:
                if is_goal(puzzle, next_node.state):
                    return solution(next_node)

                push(wait_stack, next_node)
                in_stack.add(next_node.state)
    return None

### Bài toán 8-Puzzle

In [3]:
def find_blank(cur_board):
    for row_idx in range(3):
        for col_idx in range(3):
            if cur_board[row_idx][col_idx] == 0:
                return row_idx, col_idx

def actions_8puzzle(cur_board):
    blank_r, blank_c = find_blank(cur_board)
    valid_moves = []

    if blank_r > 0: valid_moves.append("Lên")
    if blank_r < 2: valid_moves.append("Xuống")
    if blank_c > 0: valid_moves.append("Trái")
    if blank_c < 2: valid_moves.append("Phải")

    return valid_moves

def result_8puzzle(cur_board, move):
    blank_r, blank_c = find_blank(cur_board)

    new_r, new_c = blank_r, blank_c
    if move == "Lên":
        new_r -= 1
    elif move == "Xuống":
        new_r += 1
    elif move == "Trái":
        new_c -= 1
    elif move == "Phải":
        new_c += 1

    grid = [list(cur_board[0]), list(cur_board[1]), list(cur_board[2])]
    grid[blank_r][blank_c] = grid[new_r][new_c]
    grid[new_r][new_c] = 0

    return (tuple(grid[0]), tuple(grid[1]), tuple(grid[2]))

def print_state(board):
    for r in range(3):
        print(f"[{board[r][0]} {board[r][1]} {board[r][2]}]")
    print()

if __name__ == "__main__":
    start_board = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    target_board = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
    puzzle = Problem(start_board, target_board, actions_8puzzle, result_8puzzle)

    steps = DFS(puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        total_moves = len(steps) - 1
        print(f"SOLUTION (Tổng số bước DFS tìm được: {total_moves})")
        if len(steps) <= 10:
            show_indices = list(range(len(steps)))
        else:
            show_indices = [0, 1, 2, -1, len(steps) - 2, len(steps) - 1]

        for step_idx in show_indices:
            if step_idx == -1:
                print(f"... (ẩn bớt {len(steps) - 5} bước ở giữa) ...\n")
                continue
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL):")
            else:
                print(f"Bước {step_idx}:")
            print_state(steps[step_idx])

SOLUTION (Tổng số bước DFS tìm được: 49284)
Bước 0:
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1:
[1 2 3]
[4 6 0]
[7 5 8]

Bước 2:
[1 2 3]
[4 6 8]
[7 5 0]

... (ẩn bớt 49280 bước ở giữa) ...

Bước 49283:
[1 2 3]
[4 5 0]
[7 8 6]

Bước 49284 (GOAL):
[1 2 3]
[4 5 6]
[7 8 0]



### Bài toán Vacuum Robot

In [2]:
def actions_vacuum(cur_state):
    return ["Hút", "Trái", "Phải"]

def result_vacuum(cur_state, move):
    pos, room_a, room_b = cur_state

    if move == "Trái":
        return ("A", room_a, room_b)
    elif move == "Phải":
        return ("B", room_a, room_b)
    elif move == "Hút":
        if pos == "A":
            return (pos, "Sạch", room_b)
        else:
            return (pos, room_a, "Sạch")

def goal_vacuum(cur_state):
    return cur_state[1] == "Sạch" and cur_state[2] == "Sạch"

def print_vacuum_state(cur_state):
    pos, room_a, room_b = cur_state
    print(f"Robot ở phòng {pos} | Phòng A: {room_a} | Phòng B: {room_b}")

if __name__ == "__main__":
    start_vacuum = ("A", "Bẩn", "Bẩn")
    vacuum_puzzle = Problem(start_vacuum, goal_vacuum, actions_vacuum, result_vacuum)

    steps = DFS(vacuum_puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print("SOLUTION")
        for step_idx in range(len(steps)):
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL):")
            else:
                print(f"Bước {step_idx}:")
            print_vacuum_state(steps[step_idx])

SOLUTION
Bước 0:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1:
Robot ở phòng B | Phòng A: Bẩn | Phòng B: Bẩn
Bước 2:
Robot ở phòng B | Phòng A: Bẩn | Phòng B: Sạch
Bước 3:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Sạch
Bước 4 (GOAL):
Robot ở phòng A | Phòng A: Sạch | Phòng B: Sạch
